# TrustSight Analytics data pipeline walkthrough
This notebook walks through the Python data pipeline that loads raw CSVs, builds customer features, and generates the attribution metrics that link suspicious logins and fraud orders back to phishing exposure.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'data' / 'raw').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'data' / 'raw').exists():
    raise RuntimeError('Cannot locate project root with data/raw')

sys.path.insert(0, str(ROOT))

from python import data_preparation as dp

print('Repository root:', ROOT)
print('Raw data directory:', ROOT / 'data' / 'raw')

Repository root: /Users/bbkar/Documents/New Projects/amazon-trust-safety-analytics
Raw data directory: /Users/bbkar/Documents/New Projects/amazon-trust-safety-analytics/data/raw


## Load the cleaned raw tables
These CSV files are the cleaned dataset inputs for TrustSight Analytics.

In [2]:
tables = dp.load_clean_tables()
for name, table in tables.items():
    print(f'{name}:', table.shape)

tables['customers'].head()

customers: (1000, 7)
messages: (25000, 12)
interactions: (23705, 7)
logins: (4181, 9)
orders: (2855, 8)
incidents: (293, 6)


,customer_id,signup_date,country,device_type,is_prime_member,lifetime_orders,risk_segment
0,C100000,2020-06-24,US,mobile,False,33,low
1,C100001,2022-06-23,UK,web,False,20,low
2,C100002,2023-09-11,ES,mobile,False,12,low
3,C100003,2023-03-07,UK,mobile,True,23,medium
4,C100004,2025-05-05,UK,mobile,True,8,low


## Build customer-level features
The feature table includes phishing exposure, suspicious login counts, order metrics, incident counts, and attribution metrics aligned to phishing exposure windows.

In [3]:
features = dp.build_customer_features(tables)
features.columns = [c for c in features.columns]
print('Feature columns:', features.columns.tolist())
features.head()

Feature columns: ['customer_id', 'country', 'device_type', 'is_prime_member', 'lifetime_orders', 'risk_segment', 'phish_messages', 'has_phishing_history', 'interactions_clicked_link', 'interactions_entered_credentials', 'interactions_ignored', 'interactions_opened', 'interactions_reported_as_phish', 'suspicious_login_events', 'suspicious_logins_after_phish', 'orders_total_value', 'orders_avg_value', 'orders_cnt', 'fraud_orders_after_phish', 'incident_count']


,customer_id,country,device_type,is_prime_member,lifetime_orders,risk_segment,phish_messages,has_phishing_history,interactions_clicked_link,interactions_entered_credentials,interactions_ignored,interactions_opened,interactions_reported_as_phish,suspicious_login_events,suspicious_logins_after_phish,orders_total_value,orders_avg_value,orders_cnt,fraud_orders_after_phish,incident_count
0,C100000,US,mobile,False,33,low,21,True,1,0,10,12,1,0.0,0.0,19.99,19.990000,1,0.0,0.0
1,C100001,UK,web,False,20,low,19,True,1,0,6,13,5,0.0,0.0,9.99,9.990000,1,0.0,0.0
2,C100002,ES,mobile,False,12,low,18,True,1,0,4,16,1,0.0,0.0,12.99,12.990000,1,0.0,0.0
3,C100003,UK,mobile,True,23,medium,20,True,4,0,5,12,1,0.0,0.0,300.96,75.240000,4,0.0,0.0
4,C100004,UK,mobile,True,8,low,21,True,2,0,4,16,1,0.0,0.0,112.97,37.656667,3,0.0,0.0


## Attribution metrics
These columns quantify how many suspicious logins and fraud orders occur after an identified phishing message for the same customer.

In [4]:
attribution_cols = [
    'customer_id',
    'phish_messages',
    'has_phishing_history',
    'suspicious_login_events',
    'suspicious_logins_after_phish',
    'fraud_orders_after_phish',
    'incident_count'
] 

features[attribution_cols].sort_values(
    ['fraud_orders_after_phish', 'suspicious_logins_after_phish', 'phish_messages'],
    ascending=[False, False, False]
).head(10)

,customer_id,phish_messages,has_phishing_history,suspicious_login_events,suspicious_logins_after_phish,fraud_orders_after_phish,incident_count
890,C100890,24,True,1.0,1.0,2.0,3.0
306,C100306,20,True,1.0,1.0,2.0,3.0
917,C100917,19,True,1.0,1.0,2.0,3.0
203,C100203,18,True,1.0,1.0,2.0,3.0
654,C100654,18,True,1.0,1.0,2.0,3.0
739,C100739,18,True,1.0,1.0,2.0,3.0
277,C100277,17,True,1.0,1.0,2.0,3.0
16,C100016,15,True,1.0,1.0,2.0,3.0
571,C100571,15,True,1.0,1.0,2.0,3.0
24,C100024,28,True,1.0,1.0,1.0,2.0


## Save the updated feature table
The pipeline now persists the enriched customer feature table back to `output/features/customer_features.csv`.

In [5]:
OUT_PATH = ROOT / 'output' / 'features' / 'customer_features.csv'
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(OUT_PATH, index=False)
print('Saved feature file to', OUT_PATH)

Saved feature file to /Users/bbkar/Documents/New Projects/amazon-trust-safety-analytics/output/features/customer_features.csv
